Pre-cleanup of GPU memory

In [ ]:
import os
import gc

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

try:
    import torch
    for _name in ["model", "processor"]:
        if _name in globals():
            try:
                del globals()[_name]
            except Exception:
                pass

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

    print("GPU cleanup completed.")
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Cleanup warning:", type(e).__name__, str(e))

print("Pre-cleanup cell completed.")

Dependency Installation (Runtime will restart automatically. After restart, run this cell once again.)

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

MARKER = Path("/content/.direct_uml_baseline_env_prepared_v2")

if not MARKER.exists():
    print("Preparing stable Colab environment. This will restart the runtime after installation.")

    uninstall_cmd = [
        sys.executable, "-m", "pip", "uninstall", "-y",
        "numpy", "scipy", "pandas", "pillow", "protobuf",
        "transformers", "tokenizers", "accelerate",
        "codecarbon", "pypdfium2", "qwen-vl-utils"
    ]
    subprocess.run(uninstall_cmd, check=False)

    install_cmd = [
        sys.executable, "-m", "pip", "install",
        "--no-cache-dir", "--force-reinstall",
        "numpy==2.0.2",
        "scipy==1.14.1",
        "pandas==2.2.2",
        "pillow==11.3.0",
        "protobuf==5.29.5",
        "transformers==4.57.1",
        "tokenizers==0.22.1",
        "accelerate==1.11.0",
        "huggingface_hub==0.36.0",
        "safetensors==0.6.2",
        "pypdfium2==4.30.0",
        "openpyxl==3.1.5",
        "codecarbon==2.8.3",
        "qwen-vl-utils==0.0.14",
        "sentencepiece==0.2.0",
        "timm==1.0.22"
    ]
    subprocess.run(install_cmd, check=True)

    MARKER.write_text("prepared", encoding="utf-8")

    print("\nEnvironment prepared. Restarting runtime now.")
    print("After restart, run Cell 2 again, then continue to Cell 3.")

    os.kill(os.getpid(), 9)

else:
    print("Environment already prepared. Continue to Cell 3.")

imports

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import re
import csv
import gc
import json
import time
import shutil
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image
import pypdfium2 as pdfium

import torch
from codecarbon import EmissionsTracker

from transformers import AutoProcessor, AutoModelForImageTextToText

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("Imports completed.")
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Configuration

In [ ]:
# Edit TARGET_MODEL_ID only when running the next VLM.

VLM_MODEL_IDS = [
    "google/gemma-3-4b-it",
    "google/gemma-3-12b-it",
    "meta-llama/Llama-3.2-11B-Vision-Instruct",
    "llava-hf/llava-v1.6-vicuna-13b-hf",
    "Qwen/Qwen2.5-VL-7B-Instruct",
]

# Run one model at a time.
# Change this value, then run Cell 8.
TARGET_MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

# Keep false so completed run folders are not overwritten.
FORCE_RERUN = False

# Same repeated run logic as your phase wise experiment.
NUM_RUNS = 3
CPU_WARMUP_SECS = 300
GPU_WARMUP_SECS = 300
COOLDOWN_SECS = 60
MEASURE_POWER_SECS = 1

# Five report subset
N_REPORTS = 5
USE_FIXED_REPORTS = True
SELECTED_REPORT_STEMS = [
    "Report_1_uml_pages",
    "Report_5_uml_pages",
    "Report_10_uml_pages",
    "Report_20_uml_pages",
    "Report_29_uml_pages",
]

# Existing result root. The baseline folder is created inside this root.
MAIN_OUTPUT_DIR = "/content/drive/MyDrive/UML_CODECARBON_PhaseWise"

# Part 1 output folder containing extracted UML-only page PDFs.
# The code also has fallback detection below if this path is absent.
UML_PAGES_PDF_DIR = "/content/drive/MyDrive/UML_CODECARBON_PhaseWise/phase1_part1_fixed/uml_pages_pdfs"

BASELINE_ROOT = os.path.join(
    MAIN_OUTPUT_DIR,
    "direct_uml_image_to_test_baseline_5reports_multi_vlm"
)

os.makedirs(BASELINE_ROOT, exist_ok=True)

def safe_model_tag(model_id: str) -> str:
    return model_id.replace("/", "__").replace(":", "_")

def detect_uml_pages_pdf_dir():
    candidates = [
        UML_PAGES_PDF_DIR,
        "/content/drive/MyDrive/UML_Codecarbon/uml_pages_pdfs",
        "/content/drive/MyDrive/UML_CODECARBON_PhaseWise/phase1_part1_fixed/uml_pages_pdfs",
        "/content/drive/MyDrive/UML_CODECARBON_PhaseWise/phase1_part1/uml_pages_pdfs",
    ]

    for c in candidates:
        if os.path.isdir(c) and len(list(Path(c).glob("*.pdf"))) > 0:
            return c

    likely_roots = [
        "/content/drive/MyDrive/UML_CODECARBON_PhaseWise",
        "/content/drive/MyDrive/UML_Codecarbon",
    ]

    for root in likely_roots:
        if os.path.isdir(root):
            for p in Path(root).rglob("uml_pages_pdfs"):
                if p.is_dir() and len(list(p.glob("*.pdf"))) > 0:
                    return str(p)

    raise RuntimeError(
        "Could not find uml_pages_pdfs folder. Please set UML_PAGES_PDF_DIR manually."
    )

UML_PAGES_PDF_DIR = detect_uml_pages_pdf_dir()

print("TARGET_MODEL_ID:", TARGET_MODEL_ID)
print("NUM_RUNS:", NUM_RUNS)
print("CPU_WARMUP_SECS:", CPU_WARMUP_SECS)
print("GPU_WARMUP_SECS:", GPU_WARMUP_SECS)
print("UML_PAGES_PDF_DIR:", UML_PAGES_PDF_DIR)
print("BASELINE_ROOT:", BASELINE_ROOT)

General Helpers

In [ ]:

# Includes output backup, JSON saving, CodeCarbon tracker, report selection, PDF-to-image rendering, warm-up, and cleanup.

def backup_if_exists(path):
    if os.path.exists(path):
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        new_name = f"{path}.old_{ts}"
        os.rename(path, new_name)
        print(f"Existing file backed up: {new_name}")

def save_json(obj, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def make_tracker(output_dir: str, project_name: str, output_file: str):
    os.makedirs(output_dir, exist_ok=True)
    backup_if_exists(os.path.join(output_dir, output_file))
    return EmissionsTracker(
        project_name=project_name,
        output_dir=output_dir,
        output_file=output_file,
        measure_power_secs=MEASURE_POWER_SECS,
        log_level="warning",
    )

def warmup_cpu(seconds=300):
    print(f"CPU warm-up for {seconds} seconds...")
    start = time.time()
    x = 0
    while time.time() - start < seconds:
        x += sum(i * i for i in range(4000))
    print("CPU warm-up finished.")

def pdf_to_page_images(pdf_path, out_dir, scale=2.0):
    """
    Render each page of a UML-only PDF to PNG images.
    This follows the same image-rendering logic as the phase-wise notebook.
    """
    os.makedirs(out_dir, exist_ok=True)

    pdf = pdfium.PdfDocument(pdf_path)
    img_paths = []

    for i in range(len(pdf)):
        page = pdf[i]
        bitmap = page.render(scale=scale)
        img = bitmap.to_pil()

        out_path = os.path.join(
            out_dir,
            f"{Path(pdf_path).stem}_page_{i+1:03d}.png"
        )
        img.save(out_path)
        img_paths.append(out_path)

    return img_paths

def select_five_reports():
    all_pdfs = sorted(Path(UML_PAGES_PDF_DIR).glob("*.pdf"))

    if len(all_pdfs) == 0:
        raise RuntimeError(f"No PDF files found in {UML_PAGES_PDF_DIR}")

    if USE_FIXED_REPORTS:
        stem_to_path = {p.stem: p for p in all_pdfs}
        selected = []
        missing = []

        for stem in SELECTED_REPORT_STEMS:
            if stem in stem_to_path:
                selected.append(stem_to_path[stem])
            else:
                missing.append(stem)

        if missing:
            print("WARNING: These fixed reports were not found:", missing)

        if len(selected) < N_REPORTS:
            used = {p.stem for p in selected}
            for p in all_pdfs:
                if p.stem not in used:
                    selected.append(p)
                if len(selected) == N_REPORTS:
                    break
    else:
        selected = all_pdfs[:N_REPORTS]

    selected = selected[:N_REPORTS]

    if len(selected) == 0:
        raise RuntimeError("No reports selected. Check UML_PAGES_PDF_DIR and SELECTED_REPORT_STEMS.")

    print("Selected reports:")
    for p in selected:
        print(" -", p.name)

    return selected

def unload_model_safely():
    """
    Clears the current model/processor from GPU memory.
    """
    global model, processor, CURRENT_MODEL_KIND, CURRENT_MODEL_DTYPE, CURRENT_MODEL_DEVICE

    for name in ["model", "processor"]:
        try:
            if name in globals():
                del globals()[name]
        except Exception:
            pass

    CURRENT_MODEL_KIND = None
    CURRENT_MODEL_DTYPE = None
    CURRENT_MODEL_DEVICE = None

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

    print("Model memory cleanup completed.")

def is_run_completed(model_id: str, run_id: int):
    model_root = os.path.join(BASELINE_ROOT, safe_model_tag(model_id))
    summary_path = os.path.join(
        model_root,
        f"run_{run_id}",
        "inference_tracking",
        f"direct_baseline_run_{run_id}_inference_summary.json"
    )

    if not os.path.exists(summary_path):
        return False

    try:
        with open(summary_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        return data.get("status") == "completed"
    except Exception:
        return False

Prompts and Test case Parser

In [ ]:
DIRECT_IMAGE_SYSTEM_PROMPT = """
You are a Senior QA Engineer.

Input: UML diagram image for ONE software project.
The image may show one UML page and may contain class, activity, sequence,
use-case, state-machine, component, deployment, object, package, or other
UML-like design information.

Your job:
- Understand the UML information visible in the image.
- Generate COMPLETE and DETAILED test cases directly from the UML image.
- Cover visible entities, actors, lifelines, classes, operations, relationships,
  flows, conditions, branches, and validations.
- Cover positive, negative, and edge cases when supported by the visible UML.
- If some small detail is missing, make a minimal reasonable assumption and proceed
  (do not get stuck; do not invent lots of new things).

STRICT RULES:
- Output PLAIN TEXT only (no JSON).
- Do NOT output placeholders like <...>, "string", or "...".
- Do NOT repeat these instructions.
- Use concrete, realistic values when needed.
- Keep module names aligned with what is visible in the UML image.
"""

DIRECT_IMAGE_USER_PROMPT_TEMPLATE = """
You will receive ONE UML page image.

Task:
1) First write PROJECT OVERVIEW (3-5 lines).
2) Then write TEST CASES.

Use only the visible UML information from this page.

OUTPUT FORMAT (STRICT):

PROJECT OVERVIEW:
(3-5 lines summarizing the system or diagram content)

----------------------------
TEST CASES
----------------------------

For EACH test case, use EXACTLY this structure:

Test Case ID: TC-<MODULE>-<NUMBER>
Title:
Module:
Source UML Pages:
Preconditions:
Test Data:
Test Steps:
Expected Result:

Rules:
- Use sequential numbering: 001, 002, 003...
- Make Module a meaningful word (no placeholders).
- Source UML Pages must be this page number: {page_number}
- Test Steps must be numbered 1., 2., 3....
- Generate only test cases grounded in the visible UML page.
"""

def parse_testcases_from_text(llm_output: str):
    """
    Same strict parser logic as the phase-wise Part 3 output format.
    Incomplete or placeholder-based test cases are dropped.
    """
    lines = [ln.rstrip() for ln in llm_output.splitlines()]
    rows = []

    current = None
    collecting_steps = False

    def finalize(tc):
        required = [
            "Test Case ID",
            "Title",
            "Module",
            "Source UML Pages",
            "Preconditions",
            "Test Data",
            "Test Steps",
            "Expected Result"
        ]

        for k in required:
            if k not in tc or not str(tc[k]).strip():
                return None

        bad_tokens = ["<", ">", "string", "..."]
        blob = " ".join(str(tc[k]) for k in required).lower()

        if any(bt in blob for bt in bad_tokens):
            return None

        return tc

    for ln in lines:
        s = ln.strip()

        if not s:
            continue

        if s.startswith("Test Case ID:"):
            if current:
                fin = finalize(current)
                if fin:
                    rows.append(fin)

            current = {
                "Test Case ID": s.replace("Test Case ID:", "").strip(),
                "Title": "",
                "Module": "",
                "Source UML Pages": "",
                "Preconditions": "",
                "Test Data": "",
                "Test Steps": "",
                "Expected Result": ""
            }
            collecting_steps = False
            continue

        if not current:
            continue

        if s.startswith("Title:"):
            current["Title"] = s.replace("Title:", "").strip()
            collecting_steps = False

        elif s.startswith("Module:"):
            current["Module"] = s.replace("Module:", "").strip()
            collecting_steps = False

        elif s.startswith("Source UML Pages:"):
            current["Source UML Pages"] = s.replace("Source UML Pages:", "").strip()
            collecting_steps = False

        elif s.startswith("Preconditions:"):
            current["Preconditions"] = s.replace("Preconditions:", "").strip()
            collecting_steps = False

        elif s.startswith("Test Data:"):
            current["Test Data"] = s.replace("Test Data:", "").strip()
            collecting_steps = False

        elif s.startswith("Test Steps:"):
            current["Test Steps"] = s.replace("Test Steps:", "").strip()
            collecting_steps = True

        elif s.startswith("Expected Result:"):
            current["Expected Result"] = s.replace("Expected Result:", "").strip()
            collecting_steps = False

        else:
            if collecting_steps:
                current["Test Steps"] = (current["Test Steps"] + "\n" + s).strip()
            else:
                if current["Expected Result"] == "":
                    if current["Test Data"]:
                        current["Test Data"] = (current["Test Data"] + " " + s).strip()
                    else:
                        current["Preconditions"] = (current["Preconditions"] + " " + s).strip()

    if current:
        fin = finalize(current)
        if fin:
            rows.append(fin)

    return rows

def renumber_testcases(rows):
    """
    Renumber after combining all page-level outputs so IDs are unique per report.
    """
    new_rows = []

    for idx, row in enumerate(rows, start=1):
        row = dict(row)
        module = str(row.get("Module", "UML")).strip()
        module_clean = re.sub(r"[^A-Za-z0-9]+", "", module)

        if not module_clean:
            module_clean = "UML"

        row["Test Case ID"] = f"TC-{module_clean}-{idx:03d}"
        new_rows.append(row)

    return new_rows

Model Loading and Image generation Functions

In [ ]:
# Supports the five VLMs used in the original study. Loads only TARGET_MODEL_ID, not all models at once.

CURRENT_MODEL_KIND = None
CURRENT_MODEL_DTYPE = None
CURRENT_MODEL_DEVICE = None

def dtype_for_model(model_id: str):
    mid = model_id.lower()

    if "gemma" in mid:
        return torch.bfloat16

    return torch.float16

def load_model_with_dtype(model_class, model_id, model_dtype):
    kwargs = dict(
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )

    try:
        return model_class.from_pretrained(
            model_id,
            dtype=model_dtype,
            **kwargs
        )
    except TypeError:
        return model_class.from_pretrained(
            model_id,
            torch_dtype=model_dtype,
            **kwargs
        )

def load_single_vlm(model_id: str):
    """
    Load only one model at a time.
    """
    global model, processor, CURRENT_MODEL_KIND, CURRENT_MODEL_DTYPE, CURRENT_MODEL_DEVICE

    unload_model_safely()

    model_dtype = dtype_for_model(model_id)
    model_id_lower = model_id.lower()

    print("\n" + "#" * 80)
    print("Loading VLM:", model_id)
    print("#" * 80)
    print("dtype:", model_dtype)

    if "llama-3.2" in model_id_lower or "mllama" in model_id_lower:
        from transformers import MllamaForConditionalGeneration
        processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
        model = load_model_with_dtype(MllamaForConditionalGeneration, model_id, model_dtype)
        CURRENT_MODEL_KIND = "mllama"

    elif "llava" in model_id_lower:
        from transformers import LlavaNextProcessor, LlavaNextForConditionalGeneration
        processor = LlavaNextProcessor.from_pretrained(model_id, trust_remote_code=True)
        model = load_model_with_dtype(LlavaNextForConditionalGeneration, model_id, model_dtype)
        CURRENT_MODEL_KIND = "llava_next"

    elif "qwen2.5-vl" in model_id_lower or "qwen2_5_vl" in model_id_lower:
        from transformers import Qwen2_5_VLForConditionalGeneration
        processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
        model = load_model_with_dtype(Qwen2_5_VLForConditionalGeneration, model_id, model_dtype)
        CURRENT_MODEL_KIND = "qwen2_5_vl"

    else:
        processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
        model = load_model_with_dtype(AutoModelForImageTextToText, model_id, model_dtype)
        CURRENT_MODEL_KIND = "generic_image_text"

    model.eval()

    CURRENT_MODEL_DTYPE = model_dtype
    CURRENT_MODEL_DEVICE = next(model.parameters()).device

    print("Loaded model kind:", CURRENT_MODEL_KIND)
    print("Loaded on device:", CURRENT_MODEL_DEVICE)

    return model, processor

def combined_direct_prompt(page_number: int):
    user_prompt = DIRECT_IMAGE_USER_PROMPT_TEMPLATE.format(page_number=page_number)
    return DIRECT_IMAGE_SYSTEM_PROMPT.strip() + "\n\n" + user_prompt.strip()

def move_inputs_to_device(inputs, device, dtype):
    for k, v in inputs.items():
        if torch.is_tensor(v):
            inputs[k] = v.to(device)

    if "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(dtype=dtype)

    if "image_grid_thw" in inputs and torch.is_tensor(inputs["image_grid_thw"]):
        inputs["image_grid_thw"] = inputs["image_grid_thw"].to(device)

    return inputs

def decode_generated(output_ids, inputs):
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[:, prompt_len:]
    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    return text

def run_vlm_image_to_testcases(image_path: str, page_number: int, max_new_tokens: int = 2200):
    """
    Directly generate test cases from a UML image page.
    """
    global model, processor, CURRENT_MODEL_KIND, CURRENT_MODEL_DTYPE, CURRENT_MODEL_DEVICE

    prompt = combined_direct_prompt(page_number)

    input_token_count = None
    output_token_count = None

    if CURRENT_MODEL_KIND == "qwen2_5_vl":
        from qwen_vl_utils import process_vision_info

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image_path},
                    {"type": "text", "text": prompt},
                ],
            }
        ]

        text_in = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        if hasattr(processor, "tokenizer") and processor.tokenizer is not None:
            input_token_count = len(processor.tokenizer.encode(text_in, add_special_tokens=False))

        image_inputs, video_inputs = process_vision_info(messages)

        inputs = processor(
            text=[text_in],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )

        inputs = move_inputs_to_device(inputs, CURRENT_MODEL_DEVICE, CURRENT_MODEL_DTYPE)

    else:
        image = Image.open(image_path).convert("RGB")

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": prompt},
                ],
            }
        ]

        text_in = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        if hasattr(processor, "tokenizer") and processor.tokenizer is not None:
            input_token_count = len(processor.tokenizer.encode(text_in, add_special_tokens=False))

        try:
            inputs = processor(
                text=[text_in],
                images=[image],
                return_tensors="pt",
            )
        except Exception:
            inputs = processor(
                text=text_in,
                images=image,
                return_tensors="pt",
            )

        inputs = move_inputs_to_device(inputs, CURRENT_MODEL_DEVICE, CURRENT_MODEL_DTYPE)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    result = decode_generated(output_ids, inputs)

    if hasattr(processor, "tokenizer") and processor.tokenizer is not None:
        output_token_count = len(processor.tokenizer.encode(result, add_special_tokens=False))

    return result, input_token_count, output_token_count

def warmup_vlm_once():
    """
    GPU/VLM warm-up, excluded from inference tracking.
    """
    warm_path = "/content/direct_baseline_warmup.png"
    img = Image.new("RGB", (512, 512), color="white")
    img.save(warm_path)

    try:
        _ = run_vlm_image_to_testcases(
            warm_path,
            page_number=1,
            max_new_tokens=32
        )
    except Exception as e:
        print("Warm-up generation warning:", type(e).__name__, str(e)[:200])

def run_baseline_warmup():
    print("\n========== DIRECT BASELINE WARM-UP START (EXCLUDED) ==========")

    warmup_cpu(CPU_WARMUP_SECS)

    print(f"GPU/VLM warm-up for {GPU_WARMUP_SECS} seconds...")
    start = time.time()

    while (time.time() - start) < GPU_WARMUP_SECS:
        warmup_vlm_once()
        time.sleep(2)

    print("GPU/VLM warm-up finished.")
    print("========== DIRECT BASELINE WARM-UP STOP ==========\n")

Run one model

In [ ]:
# Edit TARGET_MODEL_ID in Cell 4 before running this cell.
# This cell:
#   1. Loads only TARGET_MODEL_ID.
#   2. Runs 3 repeated runs.
#   3. Uses 300s CPU + 300s GPU warm-up.
#   4. Saves results in a new baseline folder.
#   5. Unloads the model after completion.


def run_direct_baseline_single_run(model_id: str, run_id: int, selected_pdfs):
    model_tag = safe_model_tag(model_id)
    model_root = os.path.join(BASELINE_ROOT, model_tag)

    run_root = os.path.join(model_root, f"run_{run_id}")
    os.makedirs(run_root, exist_ok=True)

    OUT_ROOT = os.path.join(run_root, "DIRECT_UML_IMAGE_TO_TESTCASES")
    IMG_ROOT_DIR = os.path.join(run_root, "direct_baseline_page_images")
    TRACKING_DIR = os.path.join(run_root, "inference_tracking")
    TOKEN_CSV = os.path.join(run_root, "direct_baseline_token_counts.csv")

    os.makedirs(OUT_ROOT, exist_ok=True)
    os.makedirs(IMG_ROOT_DIR, exist_ok=True)
    os.makedirs(TRACKING_DIR, exist_ok=True)

    tracker = make_tracker(
        output_dir=TRACKING_DIR,
        project_name="direct_uml_image_to_test_baseline_inference",
        output_file=f"direct_baseline_run_{run_id}_inference_emissions.csv"
    )

    print(f"\nDIRECT BASELINE RUN {run_id} INFERENCE TRACKER START")
    tracker.start()
    infer_t0 = time.time()

    summary_rows = []
    token_rows = []
    all_testcase_rows = []

    try:
        for pdf_idx, pdf_path in enumerate(selected_pdfs, start=1):
            report_name = pdf_path.stem

            print("\n" + "=" * 70)
            print(f"[{pdf_idx}/{len(selected_pdfs)}] REPORT: {pdf_path.name}")
            print("=" * 70)

            report_out_dir = os.path.join(OUT_ROOT, report_name)
            report_img_dir = os.path.join(IMG_ROOT_DIR, report_name)
            raw_page_dir = os.path.join(report_out_dir, "raw_page_outputs")

            os.makedirs(report_out_dir, exist_ok=True)
            os.makedirs(raw_page_dir, exist_ok=True)

            page_images = pdf_to_page_images(
                str(pdf_path),
                report_img_dir,
                scale=2.0
            )

            print("Rendered page images:", len(page_images))

            report_rows = []
            combined_raw_blocks = []

            for page_idx, img_path in enumerate(sorted(page_images), start=1):
                print(f"  -> Direct test generation from page {page_idx}/{len(page_images)}")

                page_output = None
                in_tok, out_tok = 0, 0
                generation_error = None

                for attempt, max_tokens in enumerate([2200, 1700, 1200], start=1):
                    try:
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()

                        print(f"     Attempt {attempt} | max_new_tokens={max_tokens}")

                        page_output, in_tok, out_tok = run_vlm_image_to_testcases(
                            image_path=img_path,
                            page_number=page_idx,
                            max_new_tokens=max_tokens
                        )
                        break

                    except torch.cuda.OutOfMemoryError as e:
                        generation_error = str(e)
                        print(f"     CUDA OOM on attempt {attempt}. Retrying with fewer tokens.")
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()
                        gc.collect()
                        time.sleep(2)

                    except Exception as e:
                        generation_error = str(e)
                        print(f"     Generation failed on attempt {attempt}: {type(e).__name__}: {e}")
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()
                        gc.collect()
                        time.sleep(2)

                if page_output is None:
                    err_path = os.path.join(
                        raw_page_dir,
                        f"{report_name}_page_{page_idx:03d}_generation_error.txt"
                    )
                    with open(err_path, "w", encoding="utf-8") as f:
                        f.write(generation_error or "Unknown generation error")

                    token_rows.append({
                        "baseline": "direct_uml_image_to_test",
                        "model_id": model_id,
                        "run_id": run_id,
                        "report": report_name,
                        "page": page_idx,
                        "input_tokens": 0,
                        "output_tokens": 0,
                        "total_tokens": 0,
                        "source_image": img_path,
                        "status": "generation_error"
                    })
                    continue

                raw_txt_path = os.path.join(
                    raw_page_dir,
                    f"{report_name}_page_{page_idx:03d}_direct_testcases.txt"
                )
                with open(raw_txt_path, "w", encoding="utf-8") as f:
                    f.write(page_output)

                combined_raw_blocks.append(
                    f"\n{'='*70}\nMODEL: {model_id}\nRUN: {run_id}\nREPORT: {report_name} | PAGE: {page_idx}\n{'='*70}\n\n{page_output}\n"
                )

                page_rows = parse_testcases_from_text(page_output)

                for row in page_rows:
                    row = dict(row)
                    row["Baseline Source"] = "direct_uml_image_to_test"
                    row["Model ID"] = model_id
                    row["Run ID"] = run_id
                    row["Report"] = report_name
                    row["Page"] = page_idx
                    row["Source Image"] = img_path

                    if not str(row.get("Source UML Pages", "")).strip():
                        row["Source UML Pages"] = str(page_idx)

                    report_rows.append(row)

                token_rows.append({
                    "baseline": "direct_uml_image_to_test",
                    "model_id": model_id,
                    "run_id": run_id,
                    "report": report_name,
                    "page": page_idx,
                    "input_tokens": in_tok,
                    "output_tokens": out_tok,
                    "total_tokens": (in_tok or 0) + (out_tok or 0),
                    "source_image": img_path,
                    "status": "ok"
                })

            combined_txt_path = os.path.join(
                report_out_dir,
                f"{report_name}_direct_baseline_testcases_raw.txt"
            )
            with open(combined_txt_path, "w", encoding="utf-8") as f:
                f.write("\n".join(combined_raw_blocks))

            report_rows = renumber_testcases(report_rows)
            report_df = pd.DataFrame(report_rows)

            csv_out = os.path.join(
                report_out_dir,
                f"{report_name}_direct_baseline_testcases.csv"
            )
            xlsx_out = os.path.join(
                report_out_dir,
                f"{report_name}_direct_baseline_testcases.xlsx"
            )

            if len(report_df) > 0:
                with open(csv_out, "w", newline="", encoding="utf-8") as f:
                    writer = csv.DictWriter(
                        f,
                        fieldnames=report_df.columns,
                        quoting=csv.QUOTE_ALL
                    )
                    writer.writeheader()

                    for _, row in report_df.iterrows():
                        writer.writerow({k: str(v) for k, v in row.items()})

                report_df.to_excel(xlsx_out, index=False)

                print(f"✅ Saved {len(report_df)} direct baseline test cases → {report_out_dir}")

                status = "ok"
                test_case_count = len(report_df)
                all_testcase_rows.extend(report_rows)

            else:
                print("⚠️ No valid test cases parsed for report")
                status = "no_parsed_cases"
                test_case_count = 0

            summary_rows.append({
                "baseline": "direct_uml_image_to_test",
                "model_id": model_id,
                "run_id": run_id,
                "report": report_name,
                "input_pdf": str(pdf_path),
                "num_pages": len(page_images),
                "test_cases": test_case_count,
                "status": status,
                "report_output_dir": report_out_dir,
                "report_csv": csv_out if test_case_count > 0 else "",
                "report_xlsx": xlsx_out if test_case_count > 0 else "",
                "combined_raw_txt": combined_txt_path,
            })

        summary_df = pd.DataFrame(summary_rows)
        token_df = pd.DataFrame(token_rows)
        all_cases_df = pd.DataFrame(all_testcase_rows)

        summary_csv = os.path.join(OUT_ROOT, "DIRECT_BASELINE_SUMMARY_5_REPORTS.csv")
        token_df.to_csv(TOKEN_CSV, index=False)
        summary_df.to_csv(summary_csv, index=False)

        all_cases_csv = os.path.join(OUT_ROOT, "ALL_DIRECT_BASELINE_TESTCASES_5_REPORTS.csv")
        all_cases_xlsx = os.path.join(OUT_ROOT, "ALL_DIRECT_BASELINE_TESTCASES_5_REPORTS.xlsx")

        if len(all_cases_df) > 0:
            all_cases_df.to_csv(all_cases_csv, index=False)
            all_cases_df.to_excel(all_cases_xlsx, index=False)

        inference_emissions = tracker.stop()
        inference_duration = time.time() - infer_t0

        inference_summary = {
            "status": "completed",
            "baseline": "direct_uml_image_to_test",
            "run_id": run_id,
            "model_id": model_id,
            "model_kind": CURRENT_MODEL_KIND,
            "input_pdf_dir": UML_PAGES_PDF_DIR,
            "selected_reports": [p.name for p in selected_pdfs],
            "num_reports": len(selected_pdfs),
            "output_root": OUT_ROOT,
            "summary_csv": summary_csv,
            "token_csv": TOKEN_CSV,
            "all_cases_csv": all_cases_csv,
            "all_cases_xlsx": all_cases_xlsx,
            "inference_duration_sec": inference_duration,
            "inference_emissions_kg": inference_emissions,
        }

        save_json(
            inference_summary,
            os.path.join(
                TRACKING_DIR,
                f"direct_baseline_run_{run_id}_inference_summary.json"
            )
        )

        print(f"\nDIRECT BASELINE RUN {run_id} INFERENCE TRACKER STOP")
        print("Summary:", summary_csv)
        print("All cases:", all_cases_csv)

        return inference_summary

    except Exception as e:
        try:
            inference_emissions = tracker.stop()
        except Exception:
            inference_emissions = None

        inference_duration = time.time() - infer_t0

        error_summary = {
            "status": "failed",
            "baseline": "direct_uml_image_to_test",
            "run_id": run_id,
            "model_id": model_id,
            "error_type": type(e).__name__,
            "error_message": str(e),
            "inference_duration_sec": inference_duration,
            "inference_emissions_kg": inference_emissions,
        }

        save_json(
            error_summary,
            os.path.join(
                TRACKING_DIR,
                f"direct_baseline_run_{run_id}_FAILED_summary.json"
            )
        )

        raise

def run_target_model_only(model_id: str):
    if model_id not in VLM_MODEL_IDS:
        raise ValueError(f"TARGET_MODEL_ID is not in VLM_MODEL_IDS: {model_id}")

    model_tag = safe_model_tag(model_id)
    model_root = os.path.join(BASELINE_ROOT, model_tag)
    os.makedirs(model_root, exist_ok=True)

    selected_pdfs = select_five_reports()

    run_ids_to_run = []

    for run_id in range(1, NUM_RUNS + 1):
        if (not FORCE_RERUN) and is_run_completed(model_id, run_id):
            print(f"Skipping completed run: {model_id} | run_{run_id}")
        else:
            run_ids_to_run.append(run_id)

    if len(run_ids_to_run) == 0:
        print("All requested runs are already completed for:", model_id)
        print("Set FORCE_RERUN=True if you want to overwrite/re-run.")
        return {
            "status": "already_completed",
            "model_id": model_id,
            "model_root": model_root,
        }

    setup_tracking_dir = os.path.join(model_root, "setup_tracking")
    setup_tracker = make_tracker(
        output_dir=setup_tracking_dir,
        project_name="direct_uml_image_to_test_baseline_setup",
        output_file="direct_baseline_setup_emissions.csv"
    )

    print("\nDIRECT BASELINE SETUP TRACKER START")
    setup_tracker.start()
    setup_t0 = time.time()

    try:
        load_single_vlm(model_id)
        setup_emissions = setup_tracker.stop()
    except Exception:
        try:
            setup_emissions = setup_tracker.stop()
        except Exception:
            setup_emissions = None
        raise

    setup_duration = time.time() - setup_t0

    save_json(
        {
            "status": "completed",
            "baseline": "direct_uml_image_to_test",
            "model_id": model_id,
            "model_kind": CURRENT_MODEL_KIND,
            "setup_duration_sec": setup_duration,
            "setup_emissions_kg": setup_emissions,
            "baseline_model_root": model_root,
            "input_pdf_dir": UML_PAGES_PDF_DIR,
        },
        os.path.join(setup_tracking_dir, "direct_baseline_setup_summary.json")
    )

    print("DIRECT BASELINE SETUP TRACKER STOP")

    run_baseline_warmup()

    run_summaries = []

    try:
        for run_id in run_ids_to_run:
            print("\n" + "=" * 80)
            print(f"{model_id} | DIRECT BASELINE RUN {run_id}/{NUM_RUNS}")
            print("=" * 80)

            run_summary = run_direct_baseline_single_run(
                model_id=model_id,
                run_id=run_id,
                selected_pdfs=selected_pdfs
            )
            run_summaries.append(run_summary)

            if run_id != run_ids_to_run[-1]:
                print(f"\nCooldown for {COOLDOWN_SECS} seconds...")
                time.sleep(COOLDOWN_SECS)

    finally:
        unload_model_safely()

    final_summary = {
        "status": "completed",
        "baseline": "direct_uml_image_to_test",
        "model_id": model_id,
        "num_runs_requested": NUM_RUNS,
        "run_ids_executed": run_ids_to_run,
        "num_reports": len(selected_pdfs),
        "selected_reports": [p.name for p in selected_pdfs],
        "baseline_model_root": model_root,
        "runs": run_summaries,
    }

    final_summary_path = os.path.join(model_root, "direct_baseline_model_final_summary.json")
    save_json(final_summary, final_summary_path)

    print("\nDONE FOR MODEL:", model_id)
    print("Final model summary:", final_summary_path)
    print("Model output root:", model_root)

    return final_summary

summary = run_target_model_only(TARGET_MODEL_ID)
summary

Aggregate summary after all models finish

In [ ]:
def aggregate_completed_baseline_results():
    rows = []

    for model_id in VLM_MODEL_IDS:
        model_tag = safe_model_tag(model_id)
        model_root = os.path.join(BASELINE_ROOT, model_tag)

        for run_id in range(1, NUM_RUNS + 1):
            summary_csv = os.path.join(
                model_root,
                f"run_{run_id}",
                "DIRECT_UML_IMAGE_TO_TESTCASES",
                "DIRECT_BASELINE_SUMMARY_5_REPORTS.csv"
            )

            token_csv = os.path.join(
                model_root,
                f"run_{run_id}",
                "direct_baseline_token_counts.csv"
            )

            emissions_json = os.path.join(
                model_root,
                f"run_{run_id}",
                "inference_tracking",
                f"direct_baseline_run_{run_id}_inference_summary.json"
            )

            if not os.path.exists(summary_csv):
                rows.append({
                    "model_id": model_id,
                    "run_id": run_id,
                    "status": "missing_summary"
                })
                continue

            df = pd.read_csv(summary_csv)
            total_cases = int(df["test_cases"].sum()) if "test_cases" in df.columns else None
            ok_reports = int((df["status"] == "ok").sum()) if "status" in df.columns else None
            total_reports = len(df)

            emissions_kg = None
            duration_sec = None

            if os.path.exists(emissions_json):
                try:
                    with open(emissions_json, "r", encoding="utf-8") as f:
                        ej = json.load(f)
                    emissions_kg = ej.get("inference_emissions_kg")
                    duration_sec = ej.get("inference_duration_sec")
                except Exception:
                    pass

            input_tokens = None
            output_tokens = None
            total_tokens = None

            if os.path.exists(token_csv):
                try:
                    tdf = pd.read_csv(token_csv)
                    if "input_tokens" in tdf.columns:
                        input_tokens = pd.to_numeric(tdf["input_tokens"], errors="coerce").sum()
                    if "output_tokens" in tdf.columns:
                        output_tokens = pd.to_numeric(tdf["output_tokens"], errors="coerce").sum()
                    if "total_tokens" in tdf.columns:
                        total_tokens = pd.to_numeric(tdf["total_tokens"], errors="coerce").sum()
                except Exception:
                    pass

            rows.append({
                "model_id": model_id,
                "run_id": run_id,
                "status": "completed",
                "reports": total_reports,
                "ok_reports": ok_reports,
                "total_test_cases": total_cases,
                "input_tokens": input_tokens,
                "output_tokens": output_tokens,
                "total_tokens": total_tokens,
                "inference_duration_sec": duration_sec,
                "inference_emissions_kg": emissions_kg,
                "summary_csv": summary_csv,
            })

    agg_df = pd.DataFrame(rows)

    out_csv = os.path.join(BASELINE_ROOT, "DIRECT_BASELINE_ALL_MODELS_RUNS_AGGREGATE_SUMMARY.csv")
    out_xlsx = os.path.join(BASELINE_ROOT, "DIRECT_BASELINE_ALL_MODELS_RUNS_AGGREGATE_SUMMARY.xlsx")

    agg_df.to_csv(out_csv, index=False)
    agg_df.to_excel(out_xlsx, index=False)

    print("Saved aggregate CSV:", out_csv)
    print("Saved aggregate XLSX:", out_xlsx)

    display(agg_df)

    return agg_df

aggregate_completed_baseline_results()